<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 2 — Su primer motor de búsqueda</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">TF-IDF + similitud coseno, implementados desde cero · sin librerías de NLP</div></div>

## Reglas de entrega

- **Repo:** suban este notebook (ya ejecutado, con sus salidas) a su repositorio de GitHub Classroom de la organización `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio:** si usaron IA en cualquier parte, declárenlo en ese archivo (qué herramienta, para qué celda, qué les dio y qué cambiaron). No declararlo se considera falta de honestidad académica.
- **Defensa oral (eliminatoria):** se les preguntará por *cualquier* celda. "Lo generó la IA" o "lo dijo el profesor" no son justificaciones válidas. Si no pueden explicar su código, no hay calificación.
- **Penalizaciones por entrega tardía:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Las celdas marcadas con `# TODO` y las preguntas en **negritas** son lo que se evalúa. El resto es andamiaje que ya viene resuelto para que enfoquen su tiempo en lo que importa.


## Objetivo

Dado el corpus que preprocesaron en el **Lab 1**, responder consultas en lenguaje libre devolviendo un **ranking por relevancia**. La regla del curso aplica: los algoritmos núcleo (TF, IDF, TF-IDF y coseno) se implementan **desde cero**. Está prohibido `TfidfVectorizer` u otra caja negra para resolver el ejercicio (solo se permite, opcionalmente, al final para *verificar*).


## 0 · Cargar el corpus procesado del Lab 1

Debe estar `corpus_procesado.json` en la misma carpeta. Si no, vuelvan a correr el Lab 1.

In [14]:
import json
import math
import re
import unicodedata
from collections import Counter
import operator
import spacy
import nltk
from nltk.stem import SnowballStemmer


# 1. Configuración del entorno y recursos esenciales
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download
    download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

# Recuperamos la lista de stopwords del Lab 1
stopwords_es = nlp.Defaults.stop_words
CONSERVAR = {'no', 'ni', 'sin'} 
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

# 2. Cargar el corpus procesado
with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)

documentos = [d['tokens'] for d in corpus]          # lista de listas de términos
print(f'{len(corpus)} documentos cargados con éxito.')
print(f'Ejemplo del documento {corpus[0]["id"]}:', documentos[0][:8])

# 3. Reutilización del pipeline de preprocesamiento del Lab 1
def normalizar(texto, quitar_acentos=True):
    texto = texto.lower()
    # Remover HTML y URLs mediante patrones básicos
    texto = re.sub(r'<[^>]+>', '', texto)
    texto = re.sub(r'https?://\S+', '', texto)
    
    if quitar_acentos:
        texto = ''.join(
            c for c in unicodedata.normalize('NFD', texto)
            if unicodedata.category(c) != 'Mn'
        )
    texto = unicodedata.normalize('NFC', texto)
    return re.sub(r'\s+', ' ', texto).strip()

def preprocesar(texto):
    """Pipeline idéntico al Lab 1 para asegurar consistencia léxica."""
    texto_norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(texto_norm)
    lemmas = [
        token.lemma_ for token in doc 
        if token.is_alpha and not token.is_space and token.text not in MIS_STOPWORDS
    ]
    return lemmas

14 documentos cargados con éxito.
Ejemplo del documento d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


**Reutilicen su `preprocesar`.** Péguenla aquí *idéntica* a la del Lab 1. Es indispensable que la consulta pase por **exactamente el mismo pipeline** que el corpus (este es el error más común: vectorizar la consulta distinto y que nada coincida).

In [15]:
stemmer = SnowballStemmer('spanish')

def tokens_stemming(texto):
    texto_norm = normalizar(texto, quitar_acentos=True)
    # Tokenizado básico por palabras usando NLTK
    tokens = nltk.word_tokenize(texto_norm, language='spanish')
    
    # Filtrar puntuación y stopwords
    filtrados = [t for t in tokens if t.isalnum() and t not in MIS_STOPWORDS]
    
    # Aplicar Stemmer
    return [stemmer.stem(t) for t in filtrados]

def tokens_lemma(texto):
    texto_norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(texto_norm)
    
    # Añadimos .is_space para erradicar el problema detectado en la exploración 2.a
    lemmas = [
        token.lemma_ for token in doc 
        if token.is_alpha and not token.is_space and token.text not in MIS_STOPWORDS
    ]
    return lemmas

def preprocesar(texto):
    # Traemos de vuelta la lógica exacta del Lab 1
    texto_norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(texto_norm)
    lemmas = [
        token.lemma_ for token in doc 
        if token.is_alpha and not token.is_space and token.text not in MIS_STOPWORDS
    ]
    return lemmas

## 1 · Indexar — TF, IDF y TF-IDF desde cero

Implementen las tres funciones siguiendo las fórmulas de la clase. La estructura es la del slide de "15 líneas".
- `tf(doc)` → frecuencia **relativa**: `f(t,d) / |d|` (importancia local).
- `idf(corpus)` → `ln(N / df(t))` (rareza global). Cuenten **documentos**, no apariciones.
- `tfidf(doc, idf_)` → `tf(t,d) · idf(t)`.


In [16]:
def tf(doc):
    """doc: lista de terminos -> dict termino:tf (frecuencia relativa)."""
    if not doc:
        return {}
    conteos = Counter(doc)
    total_tokens = len(doc)
    return {termino: cuenta / total_tokens for termino, cuenta in conteos.items()}

def idf(corpus):
    """corpus: lista de docs (cada uno lista de terminos) -> dict termino:idf."""
    N = len(corpus)
    df_conteo = Counter()
    
    # df(t) = en cuántos documentos únicos aparece el término t
    for doc in corpus:
        terminos_unicos = set(doc)
        for termino in terminos_unicos:
            df_conteo[termino] += 1
            
    # Fórmula clásica: ln(N / df(t))
    idf_dict = {}
    for termino, df_t in df_conteo.items():
        idf_dict[termino] = math.log(N / df_t)
    return idf_dict

def tfidf(doc, idf_):
    """-> dict termino:peso tf-idf para un documento."""
    vector_tf = tf(doc)
    vector_tfidf = {}
    for termino, tf_val in vector_tf.items():
        # Si un término de la consulta no existe en el corpus, su IDF y peso es 0
        idf_val = idf_.get(termino, 0.0)
        vector_tfidf[termino] = tf_val * idf_val
    return vector_tfidf

# Construir el índice: un vector tf-idf (dict disperso) por documento
IDF = idf(documentos)
INDICE = [tfidf(doc, IDF) for doc in documentos]

# Sanity check requerido por la guía
top = sorted(INDICE[3].items(), key=operator.itemgetter(1), reverse=True)[:5]
print('Términos top de', corpus[3]['id'], '->', top)

Términos top de d04 -> [('gravemente', 0.1759371553076839), ('cultivo', 0.1759371553076839), ('maiz', 0.1759371553076839), ('frijol', 0.1759371553076839), ('fronterizo', 0.1759371553076839)]


## 2 · Procesar la consulta

La consulta es texto libre. Pásenla por el **mismo** `preprocesar`, conviértanla en vector con el **mismo** `IDF` del corpus (no recalculen IDF con la consulta dentro).

In [17]:
def vectorizar_consulta(texto):
    """texto libre -> vector tf-idf (dict) usando el IDF estático del corpus."""
    tokens_q = preprocesar(texto)
    tf_q = tf(tokens_q)
    
    vector_q = {}
    for termino, tf_val in tf_q.items():
        if termino in IDF:
            vector_q[termino] = tf_val * IDF[termino]
    return vector_q

# Prueba de verificación
print("Vector de la consulta 'sequia en los cultivos':")
print(vectorizar_consulta('sequia en los cultivos'))

Vector de la consulta 'sequia en los cultivos':
{'sequia': 0.9729550745276566, 'cultivo': 1.3195286648076292}


## 3 · Ranquear — similitud coseno

Implementen el coseno entre dos vectores dispersos (dicts) y la función `buscar` que devuelve el **top-k**.
$$\cos(\vec{q},\vec{d}) = \frac{\vec{q}\cdot\vec{d}}{\lVert\vec{q}\rVert\,\lVert\vec{d}\rVert}$$

In [18]:
def coseno(v1, v2):
    """Similitud coseno entre dos vectores dispersos (dicts). Maneja el caso de norma 0."""
    # Producto punto iterando únicamente sobre la intersección de llaves de los dicts
    producto_punto = sum(v1[t] * v2[t] for t in v1 if t in v2)
    
    # Magnitudes (Normas Euclidianas)
    norma_v1 = math.sqrt(sum(peso ** 2 for peso in v1.values()))
    norma_v2 = math.sqrt(sum(peso ** 2 for peso in v2.values()))
    
    if norma_v1 == 0.0 or norma_v2 == 0.0:
        return 0.0
        
    return producto_punto / (norma_v1 * norma_v2)

def buscar(consulta, k=5):
    """Devuelve los k documentos mas relevantes como (id, titulo, score)."""
    q = vectorizar_consulta(consulta)
    resultados = []
    
    for i, doc_vector in enumerate(INDICE):
        score = coseno(q, doc_vector)
        resultados.append((corpus[i]['id'], corpus[i]['titulo'], score))
        
    # Ordenar descendentemente por el score de similitud coseno
    resultados_ordenados = sorted(resultados, key=lambda x: x[2], reverse=True)
    return resultados_ordenados[:k]

# Prueba obligatoria de recuperación
print("Resultados para: 'sequia y cultivos de maiz'")
print("-" * 50)
for id_, titulo, score in buscar('sequia y cultivos de maiz'):
    print(f'{score:.3f}  {id_}  {titulo}')

Resultados para: 'sequia y cultivos de maiz'
--------------------------------------------------
0.447  d04  Sequia afecta cultivos de maiz
0.083  d02  Crisis hidrica golpea la region
0.000  d01  Lluvias provocan inundaciones en Tuxtla
0.000  d03  Cafe de Chiapas rompe record de exportacion
0.000  d05  Turismo crece en el Canon del Sumidero


## 4 · Rómpanlo

Entender dónde falla un modelo vale más que celebrarlo donde funciona. Empezamos con el caso de clase:


In [19]:
print('Consulta: "problemas de agua"')
for id_, titulo, score in buscar('problemas de agua'):
    print(f'{score:.3f}  {id_}  {titulo}')

# Observen: d13 (agua potable) sale bien, pero d02 (crisis HIDRICA) — claramente relevante —
# queda hundido o en 0. 'agua' e 'hidrico' son cadenas distintas: TF-IDF no sabe que son sinonimos.

Consulta: "problemas de agua"
0.476  d13  Restablecen servicio de agua potable
0.000  d01  Lluvias provocan inundaciones en Tuxtla
0.000  d02  Crisis hidrica golpea la region
0.000  d03  Cafe de Chiapas rompe record de exportacion
0.000  d04  Sequia afecta cultivos de maiz


**4.a** Encuentren **2 consultas propias** donde su buscador devuelva resultados malos. Para cada una: muestren la salida y **expliquen la causa** (sinonimia, polisemia, falta de contexto, preprocesamiento agresivo, etc.). Traigan estas dos consultas a la próxima clase: con ellas abriremos BM25.

In [20]:
# Consulta Fallida 1: Sinónimos tecnológicos avanzados
print('Consulta Fallida 1: "computadoras para inteligencia artificial"')
print("-" * 60)
for id_, titulo, score in buscar('computadoras para inteligencia artificial'):
    print(f'{score:.3f}  {id_}  {titulo}')

Consulta Fallida 1: "computadoras para inteligencia artificial"
------------------------------------------------------------
0.459  d07  UPCh inaugura laboratorio de IA
0.000  d01  Lluvias provocan inundaciones en Tuxtla
0.000  d02  Crisis hidrica golpea la region
0.000  d03  Cafe de Chiapas rompe record de exportacion
0.000  d04  Sequia afecta cultivos de maiz


_Causa de la falla 1:_ 
Causa de la falla 1: El documento d07 ("UPCh inaugura laboratorio de IA") es altamente relevante 
pero usa los términos "laboratorio", "GPUs" y "aprendizaje automático". Al buscar "computadoras", 
el modelo matemático de coincidencia exacta de caracteres falla debido a que TF-IDF carece 
de conocimiento de sinonimia contextual o semántica en el dominio computacional.

In [21]:
print('Consulta Fallida 2: "servicios financieros de origen artesanal"')
print("-" * 60)
for id_, titulo, score in buscar('servicios financieros de origen artesanal'):
    print(f'{score:.3f}  {id_}  {titulo}')

Consulta Fallida 2: "servicios financieros de origen artesanal"
------------------------------------------------------------
0.288  d08  Repunta la produccion de cacao
0.105  d06  Sismo de magnitud 5.1 frente a las costas
0.097  d09  San Cristobal, destino cultural
0.090  d13  Restablecen servicio de agua potable
0.000  d01  Lluvias provocan inundaciones en Tuxtla


_Causa de la falla 2:_ 
Causa de la falla 2: El buscador rankea en primer lugar el documento d08 ("Repunta la producción de cacao") 
debido a los términos "origen" y "artesanal". Sin embargo, la intención del usuario era un tema económico/bancario. 
El enfoque Bag-of-Words del modelo evalúa los tokens de forma independiente aislándolos del contexto global, 
siendo incapaz de notar que "origen artesanal" pertenecía aquí a la agricultura de lujo y no a las finanzas.

## 5 · (Opcional) Verificación contra scikit-learn

Solo para **comprobar** que su implementación desde cero es correcta — no sustituye al ejercicio ni cuenta como entrega. Si sus rankings coinciden a grandes rasgos con los de `TfidfVectorizer`, van bien.

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
docs_txt = [' '.join(t) for t in documentos]
vec = TfidfVectorizer()
X = vec.fit_transform(docs_txt)
q = vec.transform([' '.join(preprocesar('sequia y cultivos de maiz'))])
sims = cosine_similarity(q, X)[0]
print(sorted(zip(sims, [d['id'] for d in corpus]), reverse=True)[:5])

[(np.float64(0.44721359549995804), 'd04'), (np.float64(0.10729358282092165), 'd02'), (np.float64(0.0), 'd14'), (np.float64(0.0), 'd13'), (np.float64(0.0), 'd12')]


## Entregables del Lab 2

- [ ] `tf`, `idf`, `tfidf`, `coseno`, `vectorizar_consulta` y `buscar` implementadas desde cero.
- [ ] El índice construido y la prueba de `buscar` ejecutada con salida.
- [ ] La demostración de la falla "problemas de agua" vs. "crisis hídrica" ejecutada.
- [ ] **2 consultas fallidas propias** con su causa explicada por escrito.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
